# Cleaning Downloaded Data from avian-flu

Author: Alexander Maksiaev

Purpose: Clean downloaded data from avian-flu, rename sequences according to convention, de-duplicate from GISAID

In [1]:
# Housekeeping

import os
import glob 
import pandas as pd
import xml.etree.ElementTree as ET
import requests
import time
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 


# Dates
start_date = "06-14-2025"
end_date = "06-20-2025"
date_range = start_date + "--" + end_date
# update_date = "06-18-2025"

# Make sure you have the correct paths

home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/"
downloads = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
# home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/"
# downloads = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
references = home + "references/"
originals = downloads + "Andersen/"
saved = originals + "saved/"
temp_files = originals + "temp/"
complete_files = downloads + "complete/"

os.chdir(downloads)

## Read Metadata 

In [2]:
# Read metadata

# os.chdir(saved)
# metadata_normalized = pd.read_csv("metadata_normalized.tsv", delimiter="\t") # Collection dates

metadata_folder = originals + "avian-influenza/metadata/"
os.chdir(metadata_folder)

metadata = pd.read_csv("SraRunTable_automated.csv")

print(len(metadata)) # 7397 rows

# metadata = metadata.merge(metadata_normalized, how="outer")
print(metadata.columns)

metadata["name_state"] = metadata["geo_loc_name"].apply(lambda x: x.split("/")[1] if len(x.split("/")) > 1 else x.split("/")[0])

# Find only >= last date using Release Date from metadata 
metadata["ReleaseDate"] = metadata["ReleaseDate"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
# print(metadata["ReleaseDate"])
metadata = metadata[metadata["ReleaseDate"] >= dateutil.parser.parse(start_date).strftime("%Y-%m-%d")]
# Find only <= update date using Release Date from metadata
metadata = metadata[metadata["ReleaseDate"] <= dateutil.parser.parse(end_date).strftime("%Y-%m-%d")]

print(len(metadata)) # 6053 rows between 1/1/2024 and 4/14/2025
display(metadata)
print(metadata["Library Name"])

9808
Index(['Run', 'Assay Type', 'AvgSpotLen', 'Bases', 'BioProject', 'BioSample',
       'BioSampleModel', 'Bytes', 'Center Name', 'Collection_Date', 'Consent',
       'DATASTORE filetype', 'DATASTORE provider', 'DATASTORE region',
       'Experiment', 'geo_loc_name_country', 'geo_loc_name_country_continent',
       'geo_loc_name', 'Host', 'Instrument', 'isolate', 'Library Name',
       'LibraryLayout', 'LibrarySelection', 'LibrarySource', 'Organism',
       'Platform', 'ReleaseDate', 'create_date', 'version', 'Sample Name',
       'SRA Study', 'serotype', 'isolation_source', 'BioSample Accession',
       'is_retracted', 'retraction_detection_date_utc'],
      dtype='object')
49


,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,create_date,version,Sample Name,SRA Study,serotype,isolation_source,BioSample Accession,is_retracted,retraction_detection_date_utc,name_state
9759,SRR33993023,WGS,148.15,68132847,PRJNA1102327,SAMN49104730,Viral,22640539,USDA-NVSL,2025,...,2025-06-16 16:08:28,1,25-016878-002,SRP503016,NaN,"MILK, BULK TANK",SRS25388571,False,NaN,USA
9760,SRR33993024,WGS,148.15,63105544,PRJNA1102327,SAMN49104729,Viral,21290975,USDA-NVSL,2025,...,2025-06-16 16:08:28,1,25-016866-004,SRP503016,NaN,"MILK, BULK TANK",SRS25388572,False,NaN,USA
9761,SRR33993025,WGS,148.28,46180111,PRJNA1102327,SAMN49104728,Viral,15373063,USDA-NVSL,2025,...,2025-06-16 16:08:31,1,25-016866-002,SRP503016,NaN,"MILK, BULK TANK",SRS25388570,False,NaN,USA
9762,SRR33993026,WGS,147.53,47042737,PRJNA1102327,SAMN49104727,Viral,15677039,USDA-NVSL,2025,...,2025-06-16 16:08:27,1,25-016847-007,SRP503016,NaN,"MILK, BULK TANK",SRS25388569,False,NaN,USA
9763,SRR33993027,WGS,147.76,65389924,PRJNA1102327,SAMN49104726,Viral,21557873,USDA-NVSL,2025,...,2025-06-16 16:08:41,1,25-016847-006,SRP503016,NaN,"MILK, BULK TANK",SRS25388568,False,NaN,USA
9764,SRR33993028,WGS,147.78,74054528,PRJNA1102327,SAMN49104725,Viral,24486243,USDA-NVSL,2025,...,2025-06-16 16:08:16,1,25-016847-004,SRP503016,NaN,"MILK, BULK TANK",SRS25388567,False,NaN,USA
9765,SRR33993030,WGS,147.73,54441692,PRJNA1102327,SAMN49104724,Viral,18006567,USDA-NVSL,2025,...,2025-06-16 16:08:18,1,25-016846-001,SRP503016,NaN,"MILK, BULK TANK",SRS25388564,False,NaN,USA
9766,SRR33993041,WGS,147.27,52474370,PRJNA1102327,SAMN49104723,Viral,17356512,USDA-NVSL,2025,...,2025-06-16 16:08:17,1,25-016745-002,SRP503016,NaN,"MILK, BULK TANK",SRS25388554,False,NaN,USA
9767,SRR33993043,WGS,147.96,574015012,PRJNA1102327,SAMN49104739,Viral,204462787,USDA-NVSL,2025,...,2025-06-16 16:08:35,1,25-017370-002,SRP503016,NaN,"MILK, BULK TANK",SRS25388552,False,NaN,USA
9768,SRR33993044,WGS,147.98,239374429,PRJNA1102327,SAMN49104738,Viral,85820855,USDA-NVSL,2025,...,2025-06-16 16:08:40,1,25-017369-001,SRP503016,NaN,"MILK, BULK TANK",SRS25388551,False,NaN,USA


9759    25-016878-002-original
9760    25-016866-004-original
9761    25-016866-002-original
9762    25-016847-007-original
9763    25-016847-006-original
9764    25-016847-004-original
9765    25-016846-001-original
9766    25-016745-002-original
9767    25-017370-002-original
9768    25-017369-001-original
9769    25-017364-002-original
9770    25-017364-001-original
9771    25-017354-001-original
9772        25-002509-001-tile
9773    25-017049-001-original
9774    25-016878-005-original
9775    25-016878-003-original
9776    25-016745-001-original
9777    25-016741-001-original
9778    25-017216-002-original
9779    25-017216-001-original
9780    25-017017-005-original
9781    25-017017-004-original
9782    25-017017-003-original
9783    25-016340-002-original
9784    25-016340-001-original
9785    25-017017-002-original
9786    25-017017-001-original
9787    25-016765-001-original
9788    25-016857-001-original
9789    25-017061-004-original
9790    25-017245-001-original
9791    

In [3]:
# Get list of genotypes

os.chdir(home + "references/")

# genotypes_df = pd.read_excel("genotype_key.xlsx")

# genotypes = list(genotypes_df["Genotype"])

# print(genotypes)

genotypes = ["B3.13", "D1.1"]

# genotypes = ["B3.2"] #, "B3.2", "B3.6", "B3.7", "B3.5", "A3"]

### Naming convention ###
>A/[host]/[geo_loc_name]/[isolate]/[year]|[serotype: H5N1]|[collection_date]|[host_type]|[genotype]

host_type is from manual animal reference

In metadata, we have: host, geo_loc_name, isolate, year

We need: geo_loc_name, collection_date, host_type, genotype

host = Host

geo_loc_name (primary) = geo_loc_name

geo_loc_name (secondary) = genbank_mapping.tsv > genbank_name

isolate = isolate

collection date (primary) = Collection_Date

collection date (secondary) = https://www.ncbi.nlm.nih.gov/genbank/ > BioSample (input: BioSample) > Nucleotide > [first result] > collection_date

serotype = serotype

host type = [from ref] 

genotype = [from genoflu] -- use genoflu_results.tsv

## Get genotype, specific geolocation

In [4]:
# Get genotype from genoflu_results.tsv

os.chdir(metadata_folder)

genoflu_results = pd.read_csv("genoflu_results.tsv", delimiter="\t")

genoflu_results["Run"] = genoflu_results["sample"]

metadata = metadata.merge(genoflu_results, on="Run", how="inner")
print(metadata)
# metadata = metadata[~metadata["Genotype"].str.contains('Not assigned')] # Do not include non-assigned genotypes
metadata = metadata[metadata["Genotype"].isin(genotypes)]

# Get only the genotypes we want: B3.13 and D1.1

# b313_and_d11_only = genoflu_results[(genoflu_results["Genotype"] == "B3.13") | (genoflu_results["Genotype"] == "D1.1")]
# b313_and_d11_only = b313_and_d11_only.rename(columns={"sample": "Run"})
# b313_and_d11_only = b313_and_d11_only.drop_duplicates(subset="Run", keep="last")

# metadata = metadata.merge(b313_and_d11_only, on=["Run", "Genotype"], how="inner")

print(len(metadata)) 

display(metadata)

            Run Assay Type  AvgSpotLen      Bases    BioProject     BioSample  \
0   SRR33993023        WGS      148.15   68132847  PRJNA1102327  SAMN49104730   
1   SRR33993024        WGS      148.15   63105544  PRJNA1102327  SAMN49104729   
2   SRR33993025        WGS      148.28   46180111  PRJNA1102327  SAMN49104728   
3   SRR33993026        WGS      147.53   47042737  PRJNA1102327  SAMN49104727   
4   SRR33993027        WGS      147.76   65389924  PRJNA1102327  SAMN49104726   
5   SRR33993028        WGS      147.78   74054528  PRJNA1102327  SAMN49104725   
6   SRR33993030        WGS      147.73   54441692  PRJNA1102327  SAMN49104724   
7   SRR33993041        WGS      147.27   52474370  PRJNA1102327  SAMN49104723   
8   SRR33993043        WGS      147.96  574015012  PRJNA1102327  SAMN49104739   
9   SRR33993044        WGS      147.98  239374429  PRJNA1102327  SAMN49104738   
10  SRR33993045        WGS      148.34  238793868  PRJNA1102327  SAMN49104737   
11  SRR33993046        WGS  

,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,name_state,sample,date,File Name,Genotype,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List
0,SRR33993023,WGS,148.15,68132847,PRJNA1102327,SAMN49104730,Viral,22640539,USDA-NVSL,2025,...,USA,SRR33993023,2025-06-20_06-23-56,SRR33993023.fa,B3.13,"PB1:am4, PA:ea1, NA:ea1, PB2:am2.2, HA:ea1, NP...","am4:23-001855-001:PB1, ea1:22-003707-003:PA, e...","99.34%, 98.93%, 98.79%, 98.51%, 98.30%, 99.00%...","15, 23, 17, 34, 29, 15, 8, 12",Ran on FASTA - No Coverage Report
1,SRR33993024,WGS,148.15,63105544,PRJNA1102327,SAMN49104729,Viral,21290975,USDA-NVSL,2025,...,USA,SRR33993024,2025-06-20_06-23-56,SRR33993024.fa,B3.13,"NS:am1.1, MP:ea1, PA:ea1, PB2:am2.2, NP:am8, N...","am1.1:22-010085-001:NS, ea1:22-003707-003:MP, ...","99.05%, 98.88%, 98.75%, 98.47%, 99.06%, 98.94%...","8, 11, 27, 35, 14, 15, 18, 31",Ran on FASTA - No Coverage Report
2,SRR33993025,WGS,148.28,46180111,PRJNA1102327,SAMN49104728,Viral,15373063,USDA-NVSL,2025,...,USA,SRR33993025,2025-06-20_06-23-56,SRR33993025.fa,B3.13,"NA:ea1, HA:ea1, NS:am1.1, PB2:am2.2, MP:ea1, P...","ea1:22-003707-003:NA, ea1:22-003707-003:HA, am...","98.94%, 98.12%, 99.05%, 98.47%, 98.78%, 99.21%...","15, 32, 8, 35, 12, 18, 14, 27",Ran on FASTA - No Coverage Report
3,SRR33993026,WGS,147.53,47042737,PRJNA1102327,SAMN49104727,Viral,15677039,USDA-NVSL,2025,...,USA,SRR33993026,2025-06-20_06-23-56,SRR33993026.fa,B3.13,"HA:ea1, MP:ea1, NS:am1.1, PA:ea1, NP:am8, NA:e...","ea1:22-003707-003:HA, ea1:22-003707-003:MP, am...","98.30%, 98.78%, 98.93%, 98.88%, 98.93%, 98.65%...","29, 12, 9, 24, 16, 19, 35, 32",Ran on FASTA - No Coverage Report
4,SRR33993027,WGS,147.76,65389924,PRJNA1102327,SAMN49104726,Viral,21557873,USDA-NVSL,2025,...,USA,SRR33993027,2025-06-20_06-23-56,SRR33993027.fa,B3.13,"PB2:am2.2, MP:ea1, NA:ea1, HA:ea1, NS:am1.1, N...","am2.2:22-010445-001:PB2, ea1:22-003707-003:MP,...","98.51%, 98.78%, 98.65%, 98.30%, 98.93%, 98.93%...","34, 12, 19, 29, 9, 16, 24, 16",Ran on FASTA - No Coverage Report
5,SRR33993028,WGS,147.78,74054528,PRJNA1102327,SAMN49104725,Viral,24486243,USDA-NVSL,2025,...,USA,SRR33993028,2025-06-20_06-23-56,SRR33993028.fa,B3.13,"NS:am1.1, NA:ea1, MP:ea1, PA:ea1, HA:ea1, NP:a...","am1.1:22-010085-001:NS, ea1:22-003707-003:NA, ...","98.93%, 98.72%, 98.78%, 98.88%, 98.30%, 98.93%...","9, 18, 12, 24, 29, 16, 15, 34",Ran on FASTA - No Coverage Report
7,SRR33993041,WGS,147.27,52474370,PRJNA1102327,SAMN49104723,Viral,17356512,USDA-NVSL,2025,...,USA,SRR33993041,2025-06-20_06-23-56,SRR33993041.fa,B3.13,"NP:am8, NA:ea1, NS:am1.1, MP:ea1, PA:ea1, HA:e...","am8:23-032005-001:NP, ea1:22-003707-003:NA, am...","98.93%, 98.58%, 98.69%, 98.88%, 98.88%, 98.30%...","16, 20, 11, 11, 24, 29, 34, 10",Ran on FASTA - No Coverage Report
8,SRR33993043,WGS,147.96,574015012,PRJNA1102327,SAMN49104739,Viral,204462787,USDA-NVSL,2025,...,USA,SRR33993043,2025-06-20_06-23-56,SRR33993043.fa,B3.13,"PB1:am4, MP:ea1, NA:ea1, PB2:am2.2, HA:ea1, NP...","am4:23-001855-001:PB1, ea1:22-003707-003:MP, e...","99.30%, 98.78%, 98.65%, 98.51%, 98.42%, 98.93%...","16, 12, 19, 34, 27, 16, 28, 8",Ran on FASTA - No Coverage Report
10,SRR33993045,WGS,148.34,238793868,PRJNA1102327,SAMN49104737,Viral,85355492,USDA-NVSL,2025,...,USA,SRR33993045,2025-06-20_06-23-56,SRR33993045.fa,B3.13,"NP:am8, MP:ea1, NS:am1.1, PA:ea1, PB2:am2.2, H...","am8:23-032005-001:NP, ea1:22-003707-003:MP, am...","98.86%, 98.78%, 99.05%, 98.70%, 98.47%, 98.42%...","17, 12, 8, 28, 35, 27, 15, 19",Ran on FASTA - No Coverage Report
11,SRR33993046,WGS,148.20,185103703,PRJNA1102327,SAMN49104736,Viral,66232769,USDA-NVSL,2025,...,USA,SRR33993046,2025-06-20_06-23-56,SRR33993046.fa,B3.13,"NS:am1.1, PB1:am4, NP:am8, HA:ea1, NA:ea1, PA:...","am1.1:22-010085-001:NS, am4:23-001855-001:PB1,...","99.05%, 99.34%, 98.86%, 98.42%, 98.65%, 98.70%...","8, 15, 17, 27, 19, 28, 35, 12",

In [5]:
# Get specific geolocation and name_state from genbank_mapping.tsv

genbank_mapping = pd.read_csv("genbank_mapping.tsv", delimiter="\t")
genbank_mapping["Run"] = genbank_mapping["sra_run"]
genbank_mapping = genbank_mapping.drop_duplicates(subset="Run", keep="first") # Drop duplicates
genbank_mapping["name_state"] = genbank_mapping["genbank_name"].apply(lambda x: x.split("/")[2]) # Get the name of the state

# print(genbank_mapping)

# metadata_genbank = metadata.merge(genbank_mapping, on=["Run"]) # Only include data that has states

# Get geolocation for second state attribute

os.chdir(home + "references/")
state_ref = pd.read_csv("states_ref.csv")
metadata["Geo_Location"] = metadata["name_state"].apply(lambda x: 
                                                        state_ref.loc[state_ref["Abbreviation"] == x, 'Country'].iloc[0] 
                                                        + "-" + 
                                                        x if x in state_ref["Abbreviation"].values 
                                                        else state_ref.loc[state_ref['State'].str.contains('|'.join(x.replace(', ', ' ').split(' ')), regex=True), 'Country'].iloc[0]
                                                        + "-" + 
                                                        state_ref.loc[state_ref['State'].str.contains('|'.join(x.replace(', ', ' ').split(' ')), regex=True), 'Abbreviation'].iloc[0] 
                                                        if state_ref["State"].str.contains("|".join((x.replace(", ", " ").split(" "))), regex=True).any() 
                                                        else x)

print(state_ref.loc[state_ref['State'].str.contains('|'.join("Kentucky, whatever".replace(',', ' ').split(' ')), regex=True), 'Abbreviation'])
print(state_ref['State'].str.contains('|'.join("Kentucky, whatever".replace(', ', ' ').split(' ')), regex=True))
print(metadata["name_state"])
print(metadata["Geo_Location"])
print(len(metadata))
display(metadata) # Maybe there is no state information since 3/18/2025?

0     AL
1     AK
2     AZ
3     AR
4     AS
      ..
65    ON
66    PE
67    QC
68    SK
69    YT
Name: Abbreviation, Length: 70, dtype: object
0     False
1     False
2     False
3     False
4     False
      ...  
65    False
66    False
67    False
68    False
69    False
Name: State, Length: 70, dtype: bool
0     USA
1     USA
2     USA
3     USA
4     USA
5     USA
7     USA
8     USA
10    USA
11    USA
13    USA
14    USA
15    USA
16    USA
17    USA
18    USA
19    USA
20    USA
21    USA
22    USA
23    USA
24    USA
25    USA
26    USA
27    USA
28    USA
29    USA
31    USA
33    USA
34    USA
35    USA
36    USA
37    USA
38    USA
39    USA
40    USA
41    USA
42    USA
43    USA
44    USA
45    USA
46    USA
47    USA
48    USA
Name: name_state, dtype: object
0     USA
1     USA
2     USA
3     USA
4     USA
5     USA
7     USA
8     USA
10    USA
11    USA
13    USA
14    USA
15    USA
16    USA
17    USA
18    USA
19    USA
20    USA
21    USA
22    USA
23    USA
24  

,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,sample,date,File Name,Genotype,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List,Geo_Location
0,SRR33993023,WGS,148.15,68132847,PRJNA1102327,SAMN49104730,Viral,22640539,USDA-NVSL,2025,...,SRR33993023,2025-06-20_06-23-56,SRR33993023.fa,B3.13,"PB1:am4, PA:ea1, NA:ea1, PB2:am2.2, HA:ea1, NP...","am4:23-001855-001:PB1, ea1:22-003707-003:PA, e...","99.34%, 98.93%, 98.79%, 98.51%, 98.30%, 99.00%...","15, 23, 17, 34, 29, 15, 8, 12",Ran on FASTA - No Coverage Report,USA
1,SRR33993024,WGS,148.15,63105544,PRJNA1102327,SAMN49104729,Viral,21290975,USDA-NVSL,2025,...,SRR33993024,2025-06-20_06-23-56,SRR33993024.fa,B3.13,"NS:am1.1, MP:ea1, PA:ea1, PB2:am2.2, NP:am8, N...","am1.1:22-010085-001:NS, ea1:22-003707-003:MP, ...","99.05%, 98.88%, 98.75%, 98.47%, 99.06%, 98.94%...","8, 11, 27, 35, 14, 15, 18, 31",Ran on FASTA - No Coverage Report,USA
2,SRR33993025,WGS,148.28,46180111,PRJNA1102327,SAMN49104728,Viral,15373063,USDA-NVSL,2025,...,SRR33993025,2025-06-20_06-23-56,SRR33993025.fa,B3.13,"NA:ea1, HA:ea1, NS:am1.1, PB2:am2.2, MP:ea1, P...","ea1:22-003707-003:NA, ea1:22-003707-003:HA, am...","98.94%, 98.12%, 99.05%, 98.47%, 98.78%, 99.21%...","15, 32, 8, 35, 12, 18, 14, 27",Ran on FASTA - No Coverage Report,USA
3,SRR33993026,WGS,147.53,47042737,PRJNA1102327,SAMN49104727,Viral,15677039,USDA-NVSL,2025,...,SRR33993026,2025-06-20_06-23-56,SRR33993026.fa,B3.13,"HA:ea1, MP:ea1, NS:am1.1, PA:ea1, NP:am8, NA:e...","ea1:22-003707-003:HA, ea1:22-003707-003:MP, am...","98.30%, 98.78%, 98.93%, 98.88%, 98.93%, 98.65%...","29, 12, 9, 24, 16, 19, 35, 32",Ran on FASTA - No Coverage Report,USA
4,SRR33993027,WGS,147.76,65389924,PRJNA1102327,SAMN49104726,Viral,21557873,USDA-NVSL,2025,...,SRR33993027,2025-06-20_06-23-56,SRR33993027.fa,B3.13,"PB2:am2.2, MP:ea1, NA:ea1, HA:ea1, NS:am1.1, N...","am2.2:22-010445-001:PB2, ea1:22-003707-003:MP,...","98.51%, 98.78%, 98.65%, 98.30%, 98.93%, 98.93%...","34, 12, 19, 29, 9, 16, 24, 16",Ran on FASTA - No Coverage Report,USA
5,SRR33993028,WGS,147.78,74054528,PRJNA1102327,SAMN49104725,Viral,24486243,USDA-NVSL,2025,...,SRR33993028,2025-06-20_06-23-56,SRR33993028.fa,B3.13,"NS:am1.1, NA:ea1, MP:ea1, PA:ea1, HA:ea1, NP:a...","am1.1:22-010085-001:NS, ea1:22-003707-003:NA, ...","98.93%, 98.72%, 98.78%, 98.88%, 98.30%, 98.93%...","9, 18, 12, 24, 29, 16, 15, 34",Ran on FASTA - No Coverage Report,USA
7,SRR33993041,WGS,147.27,52474370,PRJNA1102327,SAMN49104723,Viral,17356512,USDA-NVSL,2025,...,SRR33993041,2025-06-20_06-23-56,SRR33993041.fa,B3.13,"NP:am8, NA:ea1, NS:am1.1, MP:ea1, PA:ea1, HA:e...","am8:23-032005-001:NP, ea1:22-003707-003:NA, am...","98.93%, 98.58%, 98.69%, 98.88%, 98.88%, 98.30%...","16, 20, 11, 11, 24, 29, 34, 10",Ran on FASTA - No Coverage Report,USA
8,SRR33993043,WGS,147.96,574015012,PRJNA1102327,SAMN49104739,Viral,204462787,USDA-NVSL,2025,...,SRR33993043,2025-06-20_06-23-56,SRR33993043.fa,B3.13,"PB1:am4, MP:ea1, NA:ea1, PB2:am2.2, HA:ea1, NP...","am4:23-001855-001:PB1, ea1:22-003707-003:MP, e...","99.30%, 98.78%, 98.65%, 98.51%, 98.42%, 98.93%...","16, 12, 19, 34, 27, 16, 28, 8",Ran on FASTA - No Coverage Report,USA
10,SRR33993045,WGS,148.34,238793868,PRJNA1102327,SAMN49104737,Viral,85355492,USDA-NVSL,2025,...,SRR33993045,2025-06-20_06-23-56,SRR33993045.fa,B3.13,"NP:am8, MP:ea1, NS:am1.1, PA:ea1, PB2:am2.2, H...","am8:23-032005-001:NP, ea1:22-003707-003:MP, am...","98.86%, 98.78%, 99.05%, 98.70%, 98.47%, 98.42%...","17, 12, 8, 28, 35, 27, 15, 19",Ran on FASTA - No Coverage Report,USA
11,SRR33993046,WGS,148.20,185103703,PRJNA1102327,SAMN49104736,Viral,66232769,USDA-NVSL,2025,...,SRR33993046,2025-06-20_06-23-56,SRR33993046.fa,B3.13,"NS:am1.1, PB1:am4, NP:am8, HA:ea1, NA:ea1, PA:...","am1.1:22-010085-001:NS, am4:23-001855-001:PB1,...","99.05%, 99.34%, 98.86%, 98.42%, 98.65%, 98.70%...","8, 15, 17, 27, 19, 28, 35, 12",Ra

In [6]:
# # If no states

# metadata_genbank = metadata

# metadata_genbank["name_state"] = "USA"

# metadata_genbank["Geo_Location"] = "USA"

# display(metadata_genbank)

## Collection Dates

If date is N/A, try finding it first

In [7]:

# # Get all dates
# metadata["Collection_Date_Specific"] = metadata["BioSample"].apply(lambda x: search_collection_date(x, metadata) if "-" not in x else x)

# # Save this so we don't have to do it again

# # os.chdir(temp_files)
# metadata.to_csv("metadata_genbank.csv")

In [8]:
# os.chdir(saved)
# metadata = pd.read_csv("metadata_genbank_" + date_range + ".csv")
metadata = pd.read_csv("metadata_genbank.csv")

In [9]:
date = "2025" 

date = dateutil.parser.parse(date, default=datetime(2000, 1, 1))

print(date.month)

1


In [10]:
# # # Upload saved data -- if doing this, make sure the above cell is commented out
# # os.chdir(temp_files + "saved/")
# # metadata_genbank = pd.read_csv("metadata_genbank.csv")
# # os.chdir(temp_files)

# # Get only updated dates

# # unknown_dates = metadata[(metadata["Collection_Date"] == "2024") | (metadata["Collection_Date"] == "2025")] # Dates we don't have

# def find_known_dates(x, df):
    
#     try:
#         date = metadata[metadata["BioSample"] == x]["Collection_Date"].values[0]
#     # print(date)
#         # print(date)
#         # if len(str(date)) == 4: # If this is just a year
#         #     date = search_collection_date(x, df)
#         # else:
#         date = dateutil.parser.parse(date, default=datetime(2000, 1, 1)) # .strftime("%Y-%m-%d") # If a date already exists
#         if date.day == dateutil.parser.parse("1/1/2000").day and date.month == dateutil.parser.parse("1/1/2000").month:
#             print("year only")
#             date = search_collection_date(x, df)
#         print("Success", date)
#     except:
#         date = search_collection_date(x, df) # If it's not parseable as a date
#     return date # If statement in lambda function will search for the "just year" values
        

# # years = ["2021", "2022", "2023", "2024", "2025"]
# # unknown_dates = metadata[metadata["Collection_Date"].isin(years)]
# # # known_dates = metadata[(metadata["Collection_Date"] != "2024") & (metadata["Collection_Date"] != "2025")] # Dates we've already gotten
# # known_dates = metadata[~metadata["Collection_Date"].isin(years)]

# # Get new dates also 
# # new_dates = metadata["BioSample"].apply(lambda x: search_collection_date(x, metadata_genbank) if )

# updated_dates = metadata["BioSample"].apply(lambda x: find_known_dates(x, metadata)) # Update unknown dates, if possible
# metadata["Collection_Date"] = updated_dates

# # metadata = pd.concat([known_dates, unknown_dates], ignore_index=True, sort=True)

# # updated_unknown_dates = unknown_dates["BioSample"].apply(lambda x: search_collection_date(x, unknown_dates)) # Update unknown dates, if possible
# # unknown_dates["Collection_Date"] = updated_unknown_dates

# # metadata = pd.concat([known_dates, unknown_dates], ignore_index=True, sort=True)

# # Save this so we don't have to do it again

# # os.chdir(temp_files)
# # metadata.to_csv("metadata_genbank_" + date_range + ".csv")

# # display(metadata)

In [11]:
# os.chdir(temp_files)
# metadata.to_csv("metadata_genbank_" + date_range + ".csv")

# If no collection dates

# metadata_genbank["Collection_Date_Specific"] = metadata_genbank["Collection_Date"]

## Get host type

In [12]:
# create a mask, where is True if the host does not exist
print(metadata["Host"])

mask = metadata["Host"].isna()

# choose between the original value and split isolate using the mask
metadata["Host"] = np.where(mask, metadata["isolate"].apply(lambda x: x if x != x or "/" not in x or len(x.split("/")) < 2 else x.split("/")[1]), metadata["Host"]) #  if "/" in metadata["isolate"] else metadata["Host"])
metadata["Host"] = metadata["Host"].apply(lambda x: x.lower() if x == x else x)

# print(metadata["Host"])

# Create animals ref if needed

unique_animals_all = sort_animals_andersen(metadata)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

# print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

os.chdir(home + "references/")

animals_ref = pd.read_csv("animals_ref.csv") # Upload animals ref

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer, deal with that later

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort2.csv") # Make sure name is different to avoid overwriting the first reference 

print(metadata["Host"])
# print(metadata["isolate"])

0        TURKEY VULTURE
1        TURKEY VULTURE
2       RED-TAILED HAWK
3       RED-TAILED HAWK
4      GREAT HORNED OWL
             ...       
106              CATTLE
107              CATTLE
108              CATTLE
109              CATTLE
110              CATTLE
Name: Host, Length: 111, dtype: object
[]
                  avian               cattle        feline   other_mammal  \
0      great_horned_owl            dairy_cow           cat     deer mouse   
1          common_raven               cattle  domestic_cat    house_mouse   
2         cooper's_hawk  cattle milk product     feral_cat          skunk   
3          coopers_hawk          bovine_milk        feline  striped_skunk   
4               peafowl              bovine   domestic-cat     norway rat   
..                  ...                  ...           ...            ...   
466         anser anser                  NaN           NaN            NaN   
467    northern harrier                  NaN           NaN            NaN   
4

In [13]:
# Get animals from animal reference
os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")
fix_animals_andersen(metadata, animals_ref) # Get host type

metadata["years"] = metadata["Collection_Date"].apply(lambda x: str(x).split("-")[0]) # Get year only from collection date

## Make names using all the attributes we collected

In [14]:
for num, collection_date in enumerate(metadata["Collection_Date"]):
    if collection_date != collection_date: # if nan
         collection_date = "missing"
    try:
        print(collection_date)
        if collection_date.month == datetime.parser.parse("1/1/2000").month and collection_date.day == datetime.parser.parse("1/1/2000").day:
            # If not a valid collection date, but has year
            
            year = collection_date.year
            metadata.loc[num, "Collection_Date"] = year
            # if collection_date != collection_date: # If nan
        #     metadata.loc[num, "Collection_Date"] = metadata.loc[num, "years"]
        else: # If actual date
                
            # if len(str(collection_date)) == 4: # If it's a year
                # print("caught")
                metadata.loc[num, "Collection_Date"] = collection_date
    except:
         metadata.loc[num, "Collection_Date"] = collection_date
        # else:
        #     try:
        #         parsed_date = dateutil.parser.parse(collection_date)
        #         date = parsed_date.strftime("%Y-%m-%d") # Make sure it doesn't default to today, if just a year
        #         metadata.loc[num, "Collection_Date"] = date
        #     except: # If no date at all
        #         metadata.loc[num, "Collection_Date"] = collection_date

    # metadata = metadata.dropna(thresh=2)



2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2024
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2024
2025
2025
2025
2025
2025
2025
2025
2025
2025
2024
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2024
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2024
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2024
2025
2025
2025
2025
2025
2025
2025
2024
2024


In [15]:
# Make names

metadata = metadata.fillna("")

# + metadata["BioSample"] + "|" 
names = ">" + metadata["Run"] + "|" + "A/" + metadata["Host"] + "/" + metadata["name_state"] + "/" + metadata["Library Name"] + "/" + metadata["years"].apply(lambda x: str(x)) + "|" + metadata["serotype"] + "|" + metadata["Geo_Location"] + "|" + metadata["Collection_Date"].apply(lambda x: str(x)) + "|" + metadata["Host_Type"] + "|" + metadata["Genotype"]

metadata["Name"] = names

# metadata_genbank.to_csv("metadata_genbank_named.csv")

display(metadata["Name"])

0      >SRR33830324|A/turkey vulture/USA/25-013422-00...
1      >SRR33830325|A/turkey vulture/USA/25-015561-00...
2      >SRR33830326|A/red-tailed hawk/USA/25-013427-0...
3      >SRR33830327|A/red-tailed hawk/USA/25-013424-0...
4      >SRR33830328|A/great horned owl/USA/25-015870-...
                             ...                        
106    >SRR33830443|A/cattle/USA/25-015681-003-origin...
107    >SRR33830444|A/cattle/USA/25-015681-002-origin...
108    >SRR33830445|A/cattle/USA/25-015681-001-origin...
109    >SRR33830449|A/cattle/USA/24-036379-001-tile/2...
110    >SRR33830450|A/cattle/USA/24-034788-001-tile/2...
Name: Name, Length: 111, dtype: object

In [16]:
# Drop duplicate runs 
metadata = metadata.drop_duplicates(subset="Run", keep="first")

In [17]:
print(metadata)

     Unnamed: 0          Run Assay Type  AvgSpotLen      Bases    BioProject  \
0             0  SRR33830324        WGS      147.97   74609244  PRJNA1207547   
1             1  SRR33830325        WGS      148.64  156899568  PRJNA1207547   
2             2  SRR33830326        WGS      137.62     616674  PRJNA1207547   
3             3  SRR33830327        WGS      148.23   57621808  PRJNA1207547   
4             4  SRR33830328        WGS      148.93  140872576  PRJNA1207547   
..          ...          ...        ...         ...        ...           ...   
106         109  SRR33830443        WGS      148.11  100597029  PRJNA1102327   
107         110  SRR33830444        WGS      148.50  131987248  PRJNA1102327   
108         111  SRR33830445        WGS      148.12  115677479  PRJNA1102327   
109         115  SRR33830449        WGS      131.52   67358891  PRJNA1102327   
110         116  SRR33830450        WGS      131.13   69602732  PRJNA1102327   

        BioSample BioSampleModel     By

## Make FASTA files

In [18]:
# Get information to create the fasta files

fasta_folder = originals + "avian-influenza/fasta/"

os.chdir(fasta_folder)

segments = ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]
pairs = []
fasta_files = {}

for genotype in genotypes: # ["B3.13", "D1.1"]:
    for segment in segments:
        pair = genotype + "_" + segment
        pairs.append(pair)

for pair in pairs:
    fasta_files[pair] = [] # List to hold fasta files

for run in metadata["Run"].values: # For each run 
    for dirpath, dirs, files in os.walk(fasta_folder): # Find the fasta file
        for file in files:
            file_name = os.path.join(dirpath, file) # Get file name
            # print(file_name)
            if run in file_name: # Note that there will be ~8 files total with that run name
                # Make a fasta file and put it in the list
                with open(file_name) as f:
                    lines = f.readlines()
                    sequence = lines[1] 
                    # Each run/segment pair has one sequence -- it's placed into a file with other run/segment pairs with the same segment and genotype
                    header = metadata[metadata["Run"] == run].loc[:, "Name"].values[0]
                    genotype = metadata[metadata["Run"] == run].loc[:, "Genotype"].values[0]
                    # print(header)
                    # print(genotype)
                    # break 
                    segment = file_name.split("_")[-2]
                    # Find the pair that corresponds to 
                    pair_name = genotype + "_" + segment
                    this_specific_fasta = []
                    for pair in pairs:
                        # print(pair)
                        # print(pair_name)
                        if pair_name == pair:
                            this_specific_fasta.append(header)
                            this_specific_fasta.append(sequence)
                            fasta_files[pair].append(this_specific_fasta)
                f.close()
        break 

In [19]:
# print(fasta_files.keys())

In [20]:
# Create fasta files 

# os.chdir(complete_files + "/B3_13_D1_1/" + date_range + "_B3_13_D1_1/")
os.chdir(originals + "complete/")
names = []
for pair in fasta_files.keys():
    # output_path = complete_files + "/B3_13_D1_1/" + date_range + "_B3_13_D1_1/" + pair + "_andersen_updated_" + update_date + ".fasta" 
    output_path = originals + "complete/" + pair + "_" + date_range + "_andersen_updated.fasta"

    output_file = open(output_path, "w")
    for item in fasta_files[pair]:
        # for item in item:
        # item = fasta_files[pair]
        try:
            name = str(item[0].values[0]) # See if this is one we didn't have a collection date for
        except:
            name = str(item[0])
        print(name)
        names.append(name)
        # First is header, second is sequence
        # print(value)
        output_file.write(name + "\n")
        output_file.write(item[1])
    output_file.close()

print(len(names)/8)

>SRR33830355|A/cattle/USA/25-016216-005-original/2025||USA|2025|cattle|B3.13
>SRR33830356|A/cattle/USA/25-016216-004-original/2025||USA|2025|cattle|B3.13
>SRR33830357|A/cattle/USA/25-016216-003-original/2025||USA|2025|cattle|B3.13
>SRR33830358|A/cattle/USA/25-016216-002-original/2025||USA|2025|cattle|B3.13
>SRR33830359|A/cattle/USA/25-016216-001-original/2025||USA|2025|cattle|B3.13
>SRR33830360|A/cattle/USA/25-016215-002-original/2025||USA|2025|cattle|B3.13
>SRR33830362|A/cattle/USA/25-016215-001-original/2025||USA|2025|cattle|B3.13
>SRR33830363|A/cattle/USA/25-016214-001-original/2025||USA|2025|cattle|B3.13
>SRR33830364|A/cattle/USA/25-016212-002-original/2025||USA|2025|cattle|B3.13
>SRR33830365|A/cattle/USA/25-016212-001-original/2025||USA|2025|cattle|B3.13
>SRR33830366|A/cattle/USA/25-016210-002-original/2025||USA|2025|cattle|B3.13
>SRR33830367|A/cattle/USA/25-016210-001-original/2025||USA|2025|cattle|B3.13
>SRR33830368|A/cattle/USA/25-016209-001-original/2025||USA|2025|cattle|B3.13

## De-Duplication

In [21]:
# De-duplication 

# gisaid = downloads + "GISAID/complete/all_genotypes/11-01-2021--06-13-2025_all_genotypes_Antarctica_North_America_South_America/"
    # gisaid = downloads + "Cats/Datasets/GISAID/"

for genotype in genotypes:

    # gisaid = downloads + "GISAID/complete/" + genotype.replace(".", "_") + "/" + date_range + "_" + genotype.replace(".", "_") + "_Antarctica_North_America_South_America/"
    
    gisaid = downloads + "GISAID/complete/2025-06-14--2025-06-20_" + genotype.replace(".", "_") + "_Antarctica_North_America_South_America/"

    os.chdir(gisaid)

    # Gisaid 
    dfs_gisaid_list = []
    dfs_gisaid = create_dataframes(gisaid)
    dfs_gisaid_list.append(dfs_gisaid)

# dfs_gisaid = {}
# for df_gisaid in dfs_gisaid_list:
#     dfs_gisaid = dfs_gisaid | df_gisaid
# # dfs_gisaid2 = create_dataframes(gisaid2)

B3.13_HA
B3.13_HA
B3.13_MP
B3.13_MP
B3.13_NA
B3.13_NA
B3.13_NP
B3.13_NP
B3.13_NS
B3.13_NS
B3.13_PA
B3.13_PA
B3.13_PB1
B3.13_PB1
B3.13_PB2
B3.13_PB2
D1.1_HA
D1.1_HA
D1.1_MP
D1.1_MP
D1.1_NA
D1.1_NA
D1.1_NP
D1.1_NP
D1.1_NS
D1.1_NS
D1.1_PA
D1.1_PA
D1.1_PB1
D1.1_PB1
D1.1_PB2
D1.1_PB2


In [22]:
# Do the same with Andersen 

# dfs_andersen = create_dataframes(complete_files + "/B3_13_D1_1/" + date_range + "_B3_13_D1_1/")
dfs_andersen = create_dataframes(originals + "complete/")

A1_HA
A1_MP
A1_NA
A1_NP
A1_NS
A1_PA
A1_PB1
A1_PB2
A2_HA
A2_MP
A2_NA
A2_NP
A2_NS
A2_PA
A2_PB1
A2_PB2
A3_HA
A3_MP
A3_NA
A3_NP
A3_NS
A3_PA
A3_PB1
A3_PB2
A4_HA
A4_MP
A4_NA
A4_NP
A4_NS
A4_PA
A4_PB1
A4_PB2
A5_HA
A5_MP
A5_NA
A5_NP
A5_NS
A5_PA
A5_PB1
A5_PB2
A6_HA
A6_MP
A6_NA
A6_NP
A6_NS
A6_PA
A6_PB1
A6_PB2
B1.1_HA
B1.1_MP
B1.1_NA
B1.1_NP
B1.1_NS
B1.1_PA
B1.1_PB1
B1.1_PB2
B1.2_HA
B1.2_MP
B1.2_NA
B1.2_NP
B1.2_NS
B1.2_PA
B1.2_PB1
B1.2_PB2
B1.3_HA
B1.3_MP
B1.3_NA
B1.3_NP
B1.3_NS
B1.3_PA
B1.3_PB1
B1.3_PB2
B2.1_HA
B2.1_MP
B2.1_NA
B2.1_NP
B2.1_NS
B2.1_PA
B2.1_PB1
B2.1_PB2
B2.2_HA
B2.2_MP
B2.2_NA
B2.2_NP
B2.2_NS
B2.2_PA
B2.2_PB1
B2.2_PB2
B3.10_HA
B3.10_MP
B3.10_NA
B3.10_NP
B3.10_NS
B3.10_PA
B3.10_PB1
B3.10_PB2
B3.11_HA
B3.11_MP
B3.11_NA
B3.11_NP
B3.11_NS
B3.11_PA
B3.11_PB1
B3.11_PB2
B3.12_HA
B3.12_MP
B3.12_NA
B3.12_NP
B3.12_NS
B3.12_PA
B3.12_PB1
B3.12_PB2
B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
B3.1_HA
B3.1_MP
B3.1_NA
B3.1_NP
B3.1_NS
B3.1_PA
B3.1_PB1
B3.1_PB2
B3.2_HA


In [23]:
# os.chdir(downloads)
# dfs_gisaid["B3.13_HA"].to_csv

In [24]:
for key in dfs_andersen.keys():
    dataframes = dfs_andersen[key]
    print(key)

print(dfs_andersen)

A1_HA
A1_MP
A1_NA
A1_NP
A1_NS
A1_PA
A1_PB1
A1_PB2
A2_HA
A2_MP
A2_NA
A2_NP
A2_NS
A2_PA
A2_PB1
A2_PB2
A3_HA
A3_MP
A3_NA
A3_NP
A3_NS
A3_PA
A3_PB1
A3_PB2
A4_HA
A4_MP
A4_NA
A4_NP
A4_NS
A4_PA
A4_PB1
A4_PB2
A5_HA
A5_MP
A5_NA
A5_NP
A5_NS
A5_PA
A5_PB1
A5_PB2
A6_HA
A6_MP
A6_NA
A6_NP
A6_NS
A6_PA
A6_PB1
A6_PB2
B1.1_HA
B1.1_MP
B1.1_NA
B1.1_NP
B1.1_NS
B1.1_PA
B1.1_PB1
B1.1_PB2
B1.2_HA
B1.2_MP
B1.2_NA
B1.2_NP
B1.2_NS
B1.2_PA
B1.2_PB1
B1.2_PB2
B1.3_HA
B1.3_MP
B1.3_NA
B1.3_NP
B1.3_NS
B1.3_PA
B1.3_PB1
B1.3_PB2
B2.1_HA
B2.1_MP
B2.1_NA
B2.1_NP
B2.1_NS
B2.1_PA
B2.1_PB1
B2.1_PB2
B2.2_HA
B2.2_MP
B2.2_NA
B2.2_NP
B2.2_NS
B2.2_PA
B2.2_PB1
B2.2_PB2
B3.10_HA
B3.10_MP
B3.10_NA
B3.10_NP
B3.10_NS
B3.10_PA
B3.10_PB1
B3.10_PB2
B3.11_HA
B3.11_MP
B3.11_NA
B3.11_NP
B3.11_NS
B3.11_PA
B3.11_PB1
B3.11_PB2
B3.12_HA
B3.12_MP
B3.12_NA
B3.12_NP
B3.12_NS
B3.12_PA
B3.12_PB1
B3.12_PB2
B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
B3.1_HA
B3.1_MP
B3.1_NA
B3.1_NP
B3.1_NS
B3.1_PA
B3.1_PB1
B3.1_PB2
B3.2_HA


In [25]:
print(len(list(dfs_gisaid.keys())))
print(len(list(dfs_andersen.keys())))

8
968


In [26]:
# Merge dataframes and drop duplicates

full_dfs = defaultdict(list)
# same = []
# andersen = set()
# gisaid = set()

for i, andersen_key in enumerate(dfs_andersen.keys()):
    if len(dfs_andersen[andersen_key]) > 0:
        for j, gisaid_key in enumerate(dfs_gisaid.keys()):
            if andersen_key == gisaid_key:
                # same.append(gisaid_key)
        # gisaid_key = list(dfs_gisaid.keys())[i]
        # gisaid2_key = list(dfs_gisaid2.keys())[i]

                andersen_df = dfs_andersen[andersen_key][0]
                print(len(andersen_df))
                # print(andersen_df)
                gisaid_df = dfs_gisaid[gisaid_key][0]
                print(len(gisaid_df))
                # gisaid2_df = dfs_gisaid2[gisaid2_key][0]

                print(pd.concat([gisaid_df, andersen_df]).drop_duplicates())

                full_df = pd.concat([andersen_df, gisaid_df], ignore_index=True)
                print("len full df:", len(full_df))
                test = len(full_df.drop_duplicates(subset="isolate_partial"))

                dedup_df = full_df.drop_duplicates(subset="isolate_partial", keep="last")

                # print((full_df.loc[full_df.duplicated(subset="isolate_partial")]))
                # print((full_df.loc[full_df.duplicated(subset="isolate_partial")]))
                # print(full_df)
                
                print("Keeping nothing: ", test)
                
                print("len deduplicated:", len(dedup_df))
                full_dfs[andersen_key].append(dedup_df)
            # else:
                # gisaid.add(gisaid_key)
                # andersen.add(andersen_key)
    
    # break 


# print(full_dfs)
# print(len(full_dfs))
# print(319*8)
# print(len(same))
# print(len(andersen))
# print(len(gisaid))

21
16
   isolate_partial                                        full_header  \
0       011565-001  >EPI_ISL_19909395|A/dairy_cow/Nevada/25_011565...   
1       007985-001  >EPI_ISL_19909344|A/dairy_cow/Nevada/25_007985...   
2       016340-002  >EPI_ISL_20049107|A/chicken/USA/016340-002/202...   
3       017017-005  >EPI_ISL_20049106|A/chicken/USA/017017-005/202...   
4       017216-002  >EPI_ISL_20049109|A/chicken/USA/017216-002/202...   
5       017216-001  >EPI_ISL_20049108|A/chicken/USA/017216-001/202...   
6       017017-003  >EPI_ISL_20049111|A/chicken/USA/017017-003/202...   
7       017017-004  >EPI_ISL_20049110|A/chicken/USA/017017-004/202...   
8       017017-002  >EPI_ISL_20049113|A/chicken/USA/017017-002/202...   
9       016340-001  >EPI_ISL_20049112|A/chicken/USA/016340-001/202...   
10      017017-001  >EPI_ISL_20049114|A/chicken/USA/017017-001/202...   
11      012798-007  >EPI_ISL_19909343|A/duck/New_York/25-012798-00...   
12      016765-001  >EPI_ISL_20049096|A/dairy

In [27]:
# # If none in one database, only use the other and drop duplicates

# full_dfs = defaultdict(list)
# for key in dfs_andersen.keys():
#     print(key)
# # for key in ["D1.3"]:
#     dataframes = dfs_andersen[key]
#     for i, df in enumerate(dataframes):
#         print(i)
#         try:
#             full_df = df.merge(dfs_gisaid[key][i], how="outer")
#             # print(full_df)
#             full_df = full_df.drop_duplicates(subset=["isolate_partial"])
#             full_dfs[key].append(full_df)
#         except:
#             print("Failed to merge dataframes in ", key)
#             full_dfs[key].append(dataframes[i])

## Create FASTA files combining Andersen and GISAID

In [28]:
# Create FASTA files per segment

combined_files = downloads + "Combinations/GISAID_Andersen/" # B3_13_D1_1/" + date_range + "_B3_13_D1_1/"

os.chdir(combined_files)
for pair in full_dfs.keys():
    print(pair)
    output_path = combined_files + pair + "_combined_" + date_range + ".fasta" 

    output_file = open(output_path, "w")
    for item in full_dfs[pair]:
        # for item in item:
        # item = fasta_files[pair]
        for index, row in item.iterrows():
            name = item.loc[index, "full_header"]
            sequence = item.loc[index, "sequence"]
            # print(name)
        # First is header, second is sequence
        # print(value)
            output_file.write(name)
            output_file.write(sequence)
    output_file.close()

D1.1_HA
D1.1_MP
D1.1_NA
D1.1_NP
D1.1_NS
D1.1_PA
D1.1_PB1
D1.1_PB2
